# Download the Jantzen papers

This notebook tries the first 100 marked-for-use papers in `papers_searched.csv`. It discovers public PDF locations through OpenAlex and other scholarly services, verifies the downloaded bytes, and checkpoints two reports after every paper:

- `download_status.csv`: one row for every attempted paper;
- `not_downloaded.csv`: only papers for which no candidate returned a valid PDF.

Existing valid PDFs are skipped, so it is safe to stop and rerun. Publisher login pages are not treated as PDFs. Unresolved papers remain in the failure list. If you obtain a paper through a browser or library account, save it under `data/papers/jantzen/paper_<Paper ID>.pdf` and rerun the notebook.

In [1]:
import csv
import os
from pathlib import Path

from dotenv import load_dotenv

from src.data.download import DownloadConfig, download_papers, read_paper_records

In [2]:
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()
)
load_dotenv(PROJECT_ROOT / ".env")
INPUT_CSV = PROJECT_ROOT / "data/source/Jantzen_species_interaction/papers_searched.csv"
PDF_DIR = PROJECT_ROOT / "data/papers/jantzen"
REPORT_DIR = PROJECT_ROOT / "outputs/paper_downloads"
OVERRIDE_CSV = PROJECT_ROOT / "data/ready/jantzen_pdf_urls.csv"

# OpenAlex works without an account. A real email enables additional Unpaywall results.
UNPAYWALL_EMAIL = os.getenv("UNPAYWALL_EMAIL")
SEMANTIC_SCHOLAR_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY") or None

# Try the first 100 marked-for-use papers in source CSV order. Use None for all papers.
LIMIT = 100
OVERWRITE = False

In [3]:
records = read_paper_records(INPUT_CSV, only_used=True, limit=LIMIT)
{
    "selected_papers": len(records),
    "with_doi": sum(bool(record.doi) for record in records),
    "unpaywall_enabled": bool(UNPAYWALL_EMAIL),
    "first_paper_ids": [record.paper_id for record in records[:5]],
    "last_paper_ids": [record.paper_id for record in records[-5:]],
}

{'selected_papers': 100,
 'with_doi': 96,
 'unpaywall_enabled': False,
 'first_paper_ids': ['5', '13', '14', '15', '17'],
 'last_paper_ids': ['288', '289', '290', '292', '293']}

In [4]:
config = DownloadConfig(
    pdf_dir=PDF_DIR,
    report_dir=REPORT_DIR,
    override_csv=OVERRIDE_CSV,
    unpaywall_email=UNPAYWALL_EMAIL,
    semantic_scholar_api_key=SEMANTIC_SCHOLAR_API_KEY,
    timeout_seconds=45,
    max_file_bytes=100 * 1024 * 1024,
    delay_seconds=0.25,
    overwrite=OVERWRITE,
)
summary = download_papers(records, config)
summary

[1/100] paper 5: already_downloaded
[2/100] paper 13: failed
[3/100] paper 14: failed
[4/100] paper 15: failed
[5/100] paper 17: already_downloaded
[6/100] paper 19: failed
[7/100] paper 25: failed
[8/100] paper 31: failed
[9/100] paper 38: already_downloaded
[10/100] paper 41: failed
[11/100] paper 49: failed
[12/100] paper 52: failed
[13/100] paper 55: failed
[14/100] paper 70: downloaded
[15/100] paper 74: failed
[16/100] paper 83: failed
[17/100] paper 88: failed
[18/100] paper 107: failed
[19/100] paper 189: failed
[20/100] paper 192: failed
[21/100] paper 193: failed
[22/100] paper 195: failed
[23/100] paper 196: failed
[24/100] paper 197: failed
[25/100] paper 198: downloaded
[26/100] paper 199: failed
[27/100] paper 200: failed
[28/100] paper 201: failed
[29/100] paper 202: failed
[30/100] paper 204: downloaded
[31/100] paper 205: downloaded
[32/100] paper 207: failed
[33/100] paper 208: failed
[34/100] paper 209: failed
[35/100] paper 210: failed
[36/100] paper 211: failed
[37

DownloadSummary(total=100, downloaded=30, already_downloaded=3, failed=67, status_path=PosixPath('/home/com3dian/Github/Data-extraction-post-training/outputs/paper_downloads/download_status.csv'), not_downloaded_path=PosixPath('/home/com3dian/Github/Data-extraction-post-training/outputs/paper_downloads/not_downloaded.csv'))

In [5]:
selected_ids = {record.paper_id for record in records}
with summary.not_downloaded_path.open(encoding="utf-8", newline="") as handle:
    not_downloaded = [
        row for row in csv.DictReader(handle) if row["paper_id"] in selected_ids
    ]

print(f"Could not download: {len(not_downloaded)}")
not_downloaded[:10]

Could not download: 67


[{'paper_id': '13',
  'doi': '10.2307/3544056',
  'source_link': 'https://www.jstor.org/stable/3544056?origin=crossref&seq=1',
  'citation': 'Jensen, T. S. (1985). Seed-seed predator interactions of european beech, fagus silvatica and forest rodents, clethrionomys glareolus and apodemus flavicollis. Oikos, 44(1), 149. https://doi.org/10.2307/3544056',
  'status': 'failed',
  'pdf_path': '/home/com3dian/Github/Data-extraction-post-training/data/papers/jantzen/paper_13.pdf',
  'download_url': '',
  'download_source': '',
  'attempted_urls': '["source_csv: https://www.jstor.org/stable/3544056?origin=crossref&seq=1 -> NonPdfResponse: response is not a PDF (Content-Type: text/html; charset=utf-8)", "doi_resolver: https://doi.org/10.2307/3544056 -> NonPdfResponse: response is not a PDF (Content-Type: text/html; charset=utf-8)"]',
  'error': "semantic_scholar: HTTPStatusError: Client error '429 ' for url 'https://api.semanticscholar.org/graph/v1/paper/DOI%3A10.2307%2F3544056?fields=openAccess